# GammaConv-Net walkthrough

Read a de-identified daily series, inspect the three model variants, train a short example, and reload its checkpoint. The method is described in [Yang et al. (2026)](https://doi.org/10.14042/j.cnki.32.1309.2026.04.002).

The CSV files do not represent any particular basin. Calendar dates are replaced by day indices; rainfall and flow values are unchanged.

In [ ]:
from pathlib import Path
import pandas as pd
import torch

from gammaconv.data import read_data
from gammaconv.models import make_model
from gammaconv.train import Config, run
from gammaconv.predict import forecast
from gammaconv.plot import make_figures

root = Path.cwd()
if root.name == 'examples':
    root = root.parent
data = read_data(root / 'data/series_a.csv')
data.head()

## Inspect the series

`day` is an integer; rainfall and flow are numerical measurements. Invalid or missing observations are rejected, so gaps are never silently removed.

In [ ]:
data[['rain', 'flow']].describe().round(2)

In [ ]:
rows = []
for name in ['single', 'dual', 'triple']:
    model = make_model(name, hidden=16)
    rows.append({'model': name, 'parameters': sum(p.numel() for p in model.parameters())})
pd.DataFrame(rows)

## Train and inspect outputs

This three-epoch run is a functional check. It selects a 5% continuous high-flow block within the training prefix and evaluates on the later validation suffix. Block selection and scaling use the full training prefix. Run the longer command in the README for the documented fitted example.

In [ ]:
out = root / 'runs/notebook'
record = run(Config(data=str(root / 'data/series_a.csv'), out=str(out),
                    model='dual', fraction=0.05, hidden=16, epochs=3))
pd.Series(record['validation'])

## Reload the best checkpoint

Use the saved scaler and observed histories to reproduce the validation predictions. The validation suffix starts at the recorded split row; its first 30 days supply the initial history.

In [ ]:
valid = data.iloc[record['split']['split_row']:]
result = forecast(out / 'model.pt', valid)
result.head()

In [ ]:
make_figures(out, out / 'figures')
from IPython.display import Image, display
display(Image(filename=str(out / 'figures/prediction.png')))

The plotted scores describe the selected validation checkpoint. They are not an independent test estimate or the results reported in the paper. The fitted response and allocation plots are saved beside the prediction figure. The full six-fraction experiment is available through `python -m gammaconv.experiment`.